# Data Cleaning

**Authors:** Benedikt Prisett & Stijn Diemel

This notebook cleans the raw Simpsons script and episode CSVs and produces five pre-aggregated CSVs (`df_q1.csv`–`df_q5.csv`) tailored to the five questions answered in `03_visualization.ipynb`.

**Steps**

1. Load raw data
2. Filter to speaking lines
3. Join episode metadata (season, number_in_season)
4. Drop invalid / outlier rows
5. Derive helper columns
6. Canonicalize character names (collapse variants like "Homer's Brain", "Mutant Burns" into the 13 main characters)
7. Aggregate one CSV per question
8. Verify and save


## 1. Setup & Load Raw Data

Two raw CSVs from Kaggle (https://www.kaggle.com/datasets/prashant111/the-simpsons-dataset):

- `simpsons_script_lines.csv` — every script line from every episode, including non-speaking stage directions.
- `simpsons_episodes.csv` — episode-level metadata (season, number-in-season, title, rating, etc.).

We merge the two so every line knows its season and episode-within-season.


In [1]:
import pandas as pd
import re

df_episodes = pd.read_csv("data/raw_data/simpsons_episodes.csv")
df_lines = pd.read_csv("data/raw_data/simpsons_script_lines.csv", low_memory=False)

print(f"Episodes: {df_episodes.shape[0]} rows, {df_episodes.shape[1]} columns")
print(f"Script lines: {df_lines.shape[0]} rows, {df_lines.shape[1]} columns")

Episodes: 600 rows, 14 columns
Script lines: 158271 rows, 13 columns


## 2. Filter to Speaking Lines

Non-speaking rows (`speaking_line == "false"`) are stage directions and location descriptions — e.g. `(Street: ext. street - establishing - night)` or non-verbal actions like `Bart Simpson: (ANGUISHED SCREAM)`. They have no `spoken_words` and don't belong in word/sentence counts.

This drops ~26k rows.


In [2]:
before = len(df_lines)
df_lines = df_lines[df_lines["speaking_line"] == "true"].copy()
print(f"Dropped {before - len(df_lines)} non-speaking rows")
print(f"Remaining: {len(df_lines)} rows")

Dropped 26159 non-speaking rows
Remaining: 132112 rows


## 3. Join Episode Metadata

Lines reference an episode via `episode_id`, which matches `id` in the episodes table. We bring in `season` and `number_in_season` so charts can group by season and per-episode position. Title/rating are not used downstream so we leave them out.


In [3]:
df_lines = df_lines.merge(
    df_episodes[["id", "number_in_season", "season"]],
    left_on="episode_id",
    right_on="id",
    how="left",
)
df_lines = df_lines.drop(columns=["id_x", "id_y"], errors="ignore")
print(f"After join: {len(df_lines)} rows")
print(f"Seasons present: {sorted(df_lines['season'].dropna().unique().tolist())}")

After join: 132112 rows
Seasons present: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]


## 4. Drop Invalid / Outlier Rows

Three drops, each documented with its rationale:

- **`word_count > 150`** — a small number of lines have absurd word counts caused by CSV column-misalignment (e.g. timestamp values leaking into `word_count`). 150 is well above any genuine line length.
- **`season == 26`** — season 26 is incomplete in this dataset, so per-season aggregations would be misleading.
- **NaN in `raw_character_text`, `normalized_text`, or `word_count`** — rows that can't be attributed to a character or counted.


In [4]:
df_lines["word_count"] = pd.to_numeric(df_lines["word_count"], errors="coerce")

before = len(df_lines)
df_lines = df_lines[df_lines["word_count"] <= 150]
print(f"Dropped {before - len(df_lines)} rows with word_count > 150")

before = len(df_lines)
df_lines = df_lines[df_lines["season"] != 26]
print(f"Dropped {before - len(df_lines)} rows from season 26")

before = len(df_lines)
df_lines = df_lines.dropna(
    subset=["raw_character_text", "normalized_text", "word_count"]
)
print(f"Dropped {before - len(df_lines)} rows with NaN in key columns")

print(f"Remaining: {len(df_lines)} rows")

Dropped 23 rows with word_count > 150
Dropped 3374 rows from season 26
Dropped 25 rows with NaN in key columns
Remaining: 128690 rows


## 5. Derive Helper Columns

- `lower_characters` — lowercase character text, used by the canonicalization step to match name variants case-insensitively.
- `timestamp_in_min` — minute mark within the episode, derived from `timestamp_in_ms`. Used by Q4 to bucket bars per minute.
- `sentence_count` — splits `spoken_words` on `.`, `!`, `?`. Used by Q5.


In [5]:
def count_sentences(text):
    sentences = re.split(r"[.!?]+", text)
    sentences = [s for s in sentences if s.strip()]
    return max(len(sentences), 1)


df_lines["lower_characters"] = df_lines["raw_character_text"].str.lower()
df_lines["timestamp_in_ms"] = pd.to_numeric(df_lines["timestamp_in_ms"])
df_lines["timestamp_in_min"] = (df_lines["timestamp_in_ms"] // 60000) + 1
df_lines["sentence_count"] = df_lines["spoken_words"].apply(count_sentences)

df_lines.head()

,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count,number_in_season,season,lower_characters,timestamp_in_min,sentence_count
0,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,true,464,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31.0,19,2,miss hoover,15,2
1,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,true,9,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3.0,19,2,lisa simpson,15,2
2,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,true,464,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22.0,19,2,miss hoover,15,4
3,32,212,Lisa Simpson: That life is worth living.,864000,true,9,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5.0,19,2,lisa simpson,15,1
4,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,true,40,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33.0,19,2,edna krabappel-flanders,15,3


## 6. Canonicalize Character Names

The dataset has thousands of `raw_character_text` variants for the same characters: `"Homer's Brain"`, `"Mutant Burns"`, `"8-Year-Old Bart"`, `"Ghost Marge"`, etc. These should count as Homer, Burns, Bart, Marge respectively.

**Approach:**

1. Discover the top 13 main characters by raw word count.
2. Build a curated `match_names` allowlist of variant strings that should be re-mapped (curated by hand to avoid false positives like `"Skinner Body"` accidentally matching unrelated rows).
3. For each row in the allowlist, find which main character's first name appears as a substring (case-insensitive) and rewrite the name.
4. Filter `df_final` to only rows belonging to the 13 main characters — minor / one-off characters are dropped because the visualization focuses on protagonists and including everyone would clutter the charts.


In [6]:
top_characters = (
    df_lines.groupby("raw_character_text")["word_count"]
    .sum()
    .sort_values(ascending=False)
    .head(13)
    .index.tolist()
)

name_characters = [
    "Homer Simpson",
    "Marge Simpson",
    "Bart Simpson",
    "Lisa Simpson",
    "C. Montgomery Burns",
    "Moe Szyslak",
    "Seymour Skinner",
    "Ned Flanders",
    "Krusty the Clown",
    "Chief Wiggum",
    "Grampa Simpson",
    "Kent Brockman",
    "Milhouse Van Houten",
    "Apu Nahasapeemapetilon",
    "Lenny Leonard",
    "Waylon Smithers",
    "Nelson Muntz",
    "Dr. Julius Hibbert",
    "Carl Carlson",
    "Edna Krabappel-Flanders",
]

short_name_characters = [
    "Homer",
    "Marge",
    "Bart",
    "Lisa",
    "Burns",
    "Moe",
    "Skinner",
    "Flanders",
    "Krusty",
    "Wiggum",
    "Grampa",
    "Kent",
    "Milhouse",
    "Apu",
    "Lenny",
    "Smithers",
    "Nelson",
    "Hibbert",
    "Carl",
    "Edna",
]

lower_name_characters = [n.lower() for n in short_name_characters]
pattern = "|".join(lower_name_characters)
df_names = df_lines[
    (df_lines["lower_characters"].str.contains(pattern, na=False))
    & ~(df_lines["raw_character_text"].isin(name_characters))
]

# Curated allowlist of variant character strings to rewrite.
# Kept verbatim — adjusting would shift word counts.
match_names = [
    "Bart's Head",
    "ADVISORS & SMITHERS",
    "BETTY & MOEHOMER (CONT'D",
    "Skinner Body",
    "BART & LISA",
    "Homer-Ape",
    "Baby Lisa",
    "CHIF WIGGUM",
    "HOMER'S THOUGHT",
    "Skinner Zombie",
    "Zombie Krusty",
    "Zombie Flanders",
    "Lisa Snail",
    "Homer",
    "Homer's Spirit",
    "Bart Anchor",
    "Homer's Thought Bubble",
    "Homer's Bloody Skull",
    "Apu's Brain",
    "Homer's Conscience",
    "Burns' Grandfather",
    "Bart's Thoughts",
    "Lisa's Thoughts",
    "Homer's Brain",
    "Evil Homer",
    "2-Year-Old Lisa",
    "7-YEAR OLD BURNS",
    "Pieces Of Homer",
    "Flanders' Inner Child",
    "Homer's Inner Child",
    "Moe's Inner Child",
    "Triple Milhouse",
    "Jerry Lewis Bart",
    "Cherub Lenny",
    "Cherub Carl",
    "Baby Burns",
    "Teenage Burns",
    "Marge",
    "Lisa",
    "Baby Bart",
    "Lunchlady Bart",
    "Grampa's Brain",
    "Nelson Apparition",
    "7-Year-Old Homer",
    "7-YEAR-OLD-HOMER",
    "Bart's Fist",
    "Adult Nelson",
    "Imaginary Burns #1",
    "Imaginary Burns #2",
    "Imaginary Burns #3",
    "Ten Imaginary Burnses",
    "Smithers's Brain",
    "Pre-Teen Hibbert Kid",
    "Little Hibbert Girl",
    "Teenage Hibbert Boy",
    "NELSON/HIS SOUL",
    "Thistlewick Flanders",
    "Homer's Mouth",
    "Student Wiggum",
    "Phony Homer",
    "Phony Marge",
    "Young Apu",
    "Apu II",
    "Milhouse",
    "Simulated Homer",
    "Bernice Hibbert",
    "Marge's Dummy",
    "Flashback Homer",
    "Flashback Marge",
    "Homer's Recording",
    "Milhouse's Thoughts",
    "Nelson's Brain",
    "SMITHERS'",
    "Smithers's Echo",
    "Mr. Burns's Thoughts",
    "1986 Krusty",
    "Young Flanders",
    "Sgt. Skinner",
    "Krusty",
    "Mutant Dr. Hibbert",
    "Mutant Moe",
    "Mutant Skinner",
    "Mutant Burns",
    "Mutant Lenny",
    "Mutant Wiggum",
    "MUTANT FLANDERS",
    "Constable Wiggum",
    "Goodman Skinner",
    "Goodman Flanders",
    "Goodman Moe",
    "Goodman Lenny",
    "Teenage Smithers",
    "MIDDLE-AGE GRAMPA",
    "Teenage Lenny",
    "Homer's Stomach",
    "Homer's Feet",
    "Homer's Voice",
    "GHOSTLY HOMER",
    "Thought Bubble Moe",
    "Thought Bubble Bart",
    "HOMER'S VOICE MAIL",
    "Cardinal Flanders",
    "President Lenny",
    "BART BART",
    "Middle-aged Grampa",
    "3-Year-Old Homer",
    "Bad Flanders",
    "DETECTIVE HOMER SIMPSON",
    "Pharaoh Skinner",
    "Moses/milhouse",
    "Methuselah Grampa",
    "Goliath/nelson",
    "Homer Effigy",
    "LISA JUNIOR",
    "20-ISH MOE",
    "Hologram Nelson",
    "Old Krusty",
    "Latin Milhouse",
    "LISABELLA",
    "Thought Bubble Homer",
    "LISA'S LAWYER",
    "Willie Nelson",
    "Ghost Homer",
    "Virtual Wiggum",
    "Fake Homer",
    "APU+",
    "Young Skinner",
    "Camel Lisa",
    "Harem Girl Bart",
    "SIX-YEAR-OLD HOMER",
    "Sourdough Moe",
    "Sheriff Wiggum",
    "Bart's Prince",
    "KRUSTY FACE",
    "MR. BURNS BRAIN",
    "Young Lenny",
    "Young Carl",
    "Young Moe",
    "Grampa's Recorded Voice",
    "Lenny Pig",
    "Hamlet Bart",
    "Guilden-Lenny",
    "Rosen-Carl",
    "Present-Day Homer",
    "Cartoon Bart",
    "Dream Homer",
    "Homer Devil",
    "Homer Angel",
    "Homer Double",
    "TRACEY ULLMAN SHOW HOMER",
    "HOMER DOUBLES",
    "Future Homer",
    "Cow Flanders",
    "Pig Wiggum",
    "Owl Lisa",
    "Fox Burns",
    "LISA OWL",
    "Spider Bart",
    "Walrus Homer",
    "Panther Marge",
    "Drunk Homer",
    "HOMER THOUGHTS",
    "SIX-YEAR-OLD GRAMPA",
    "Skinner's Thoughts",
    "Mrs. Skinner's Thoughts",
    "FLANDERS' VOICE",
    "Sheriff Lisa",
    "Bart's Voice",
    "60-Year-Old Nelson",
    "Real Lisa",
    "Hibbert Boy",
    "Flanders's Thoughts",
    "MOE HOWARD",
    "Animated Skinner",
    "PUZZLE LENNY",
    "Young Marge",
    "Young Dr. Hibbert",
    "Homer's Thoughts",
    "HOMER-ISH WOMAN'S VOICE",
    "Knockahomer",
    "Bartleby",
    "Prince Bart",
    "Advisor Moe",
    "Chief Homer",
    "Chancellor Smithers",
    "Marge's Thoughts",
    "Skinner On Plaque",
    "CYBORG SPIDER MRS. SKINNER",
    "PRINCIPAL SKINNER CT",
    "10 Years Younger Homer",
    "10 Years Younger Marge",
    "10-Year-Old Homer",
    "10-Year-Old Lenny",
    "10-Year-Old Carl",
    "TEN-YEAR-OLD MARGE",
    "Homer's Mind",
    "Marge's Mind",
    "Inspector Wiggum",
    "Ebenezer Burns",
    "2nd Homer",
    "Marge's Face",
    "Teenage Moe",
    "Lisa's Reflection",
    "Homer's Muzzle",
    "MOE-BIRD",
    "Moe's Thoughts",
    "Mr. Burns Logo",
    "Bart on Tape",
    "Homers",
    "SIX-YEAR-OLD-BART",
    "Muscular Homer",
    "Teenage Nelson",
    "Teenage Milhouse",
    "Teenage Lisa",
    "Robo-Wiggum",
    "MOE-CLONE",
    "Spider Moe",
    "50-ish Lisa",
    "50-ish Milhouse",
    "President Homer",
    "Sergeant Skinner",
    "Bart Spider",
    "Milhouse Slug",
    "Grampa Gorilla",
    "Dracula Hibbert",
    "Einstein Lisa",
    "Homer's Head",
    "Werewolf Bart",
    "Apu Robot",
    "Moe Pacifier",
    "Lenny Pacifier",
    "Old Marge",
    "Thought Bubble Marge",
    "Younger Milhouse",
    "Hibbert Wise Man",
    "Skinner Wise Man",
    "Shepherd Carl",
    "Shepherd Lenny",
    "WISE MAN HIBBERT",
    "Young Burns",
    "Toddler Homer",
    "40-ish Grampa",
    "Fun Homer",
    "Serious Homer",
    "Bart-Jack",
    "Bart-waitress",
    "Bart Head",
    "Moe Recording",
    "Mr. Burns's Reflection",
    "Beefeater Lenny",
    "Beefeater Carl",
    "Moezekiel",
    "Pilgrim Smithers",
    "King Homer",
    "Queen Marge",
    "Capt. Burns",
    "CHIEF PETTY OFFICER WIGGUM",
    "C.P.O. Wiggum",
    "Assistant G.K. Skinner",
    "Teenage Marge",
    "Little Moe Szyslak",
    "50-Foot Lenny",
    "Invisible Carl",
    "7-Year-Old Marge",
    "Gendarme Wiggum",
    "Bart-man",
    "Poison Lenny",
    "Elf Marge",
    "11-Year-Old Moe",
    "8-Year-Old Lenny",
    "8-Year-Old Carl",
    "8-Year-Old Homer",
    "8-Year-Old Wiggum",
    "16-Year-Old Wiggum",
    "24-Year-Old Wiggum",
    "32-Year-Old Wiggum",
    "24-Year-Old Homer",
    "32-Year-Old Homer",
    "8-Year-Old Marge",
    "24-Year-Old Marge",
    "2-Year-Old Bart",
    "VENDOR APU",
    "Bird Skinner",
    "Troll Moe",
    "Wench Milhouse",
    "Nelson's Head",
    "Old Bart",
    "Old Milhouse",
    "Ghost Marge",
    "Memory Wiggum",
    "Memory Marge",
    "Memory Homer",
    "Memory Lisa",
    "Memory Bart",
    "20-Year-Old Homer",
    "Young Grampa",
    "Young Homer",
    "Moe Dog",
    "Lisa Puppy",
    "Bart Puppy",
    "Prisoner Lisa",
    "Smoke Lisa",
    "Little Homer",
    "3-Year-Old Lisa",
    "5-Year-Old Bart",
    "Thought Bubble Lenny",
    "Teenage Carl",
    "Young Smithers",
    "HOMER'S HAND",
    "German Krusty",
    "Wiggum Smiley Face",
    "Carl #2",
    "Carl #1",
    "Homer Snowman",
    "Adult Lisa",
    "Colonel Burns",
    "Thought Bubble Grampa",
    "Thought Bubble Lisa",
    "MARGE 40-YEAR-OLD",
    "Mr. Burns Heads",
    "Devil Moe",
    "Avatar Bart",
    "THOUGHT-BUBBLE FLANDERS",
    "1970s Grampa",
    "Lenny's Face",
    "Skinner's Face",
    "Moe's Face",
    "Avatar Milhouse",
    "Adult Bart",
    "Elderly Principal Skinner",
    "Adult Milhouse",
    "Aged Burns",
    "Aged Krusty",
    "Lenny",
    "Carl",
    "Bartholomé",
    "Archbishop Smithers",
    "French Wiggum",
    "Viking Lenny",
    "Viking Homer",
    "Christian Homer",
    "King Nelson",
    "Thought Bubble Apu",
    '"Shorts" Homer',
    '"Shorts" Bart',
    '"Shorts" Lisa',
    '"Shorts" Marge',
    "FLANDERS' MOUTH AND MOUSTACHE",
    "Elderly Homer",
    "Elderly Marge",
    "1-Year-Old Homer",
    "80-Year-Old Bart",
    "Elderly Nelson",
    "Homer's Reflection",
    "Older Flanders",
    "Little Marge",
    "Corrupt Pope Homer",
    "Fop Homer",
    "Egyptian Slave Homer",
    "Thought Bubble Dr. Hibbert",
    "Angel Skinner",
    "DUFFMAN VOICE MILHOUSE",
    "Stanley Kowalski Milhouse",
    "1-Year-Old Bart",
    "1-Year-Old Nelson",
    "800-Pound Homer",
    "Kent Brockman's Thoughts",
    "MR. BURNS.",
    "Rastafarian Krusty",
    "Somber Irish Krusty",
    "IRISH KRUSTY",
    "Mexican Krusty",
    "Botswana Krusty",
    "Bart Snail",
    "Animated Krusty",
    "MARGE 7",
    "ZOMBIE MILHOUSE",
    "12-Year-Old Homer",
    "Super Milhouse",
    "KRUSTY'S HEAD",
    "TEENAGED KRUSTY",
    "MIDDLE-AGED KRUSTY",
    "Krustys",
    "Caveman Homer",
    "ORIGINAL HOMER",
    "ORIGINAL BART",
    "ORIGINAL MARGE",
    "REGULAR GHOST MARGE",
    "REGULAR GHOST LISA",
    "REGULAR GHOST HOMER",
    "ORIGINAL GHOST MARGE",
    "REGULAR GHOST MARGE/ ORIGINAL GHOST MARGE",
    "ORIGINAL GHOST BART",
    "ORIGINAL GHOST HOMER",
    "CGI HOMER",
    "ORIGINAL GHOST LISA",
    "MARGE *",
    "NELSON COMET",
    "HOMER)",
    "SMITHERS' THOUGHTS",
    "AGED MOE",
    "Teenage Homer",
    "Young Krusty",
    "Teenage Bart",
    'Adult "Bart"',
    "Kirk Voice Milhouse",
    "Homer The Thief",
    "KENT",
]

replace_dict = dict(zip(lower_name_characters, name_characters))
match_set = set(match_names)


def process_name(name):
    if name in match_set:
        for lower, real in replace_dict.items():
            if lower in name.lower():
                return real
    return name


df_lines["character_names"] = df_lines["raw_character_text"].apply(process_name)

df_final = df_lines[
    [
        "character_names",
        "episode_id",
        "season",
        "number_in_season",
        "timestamp_in_ms",
        "timestamp_in_min",
        "word_count",
        "sentence_count",
    ]
]
df_final = df_final[df_final["character_names"].isin(top_characters)]

print(f"Rows after filtering to top 13 characters: {len(df_final)}")
print(f"Unique characters: {df_final['character_names'].nunique()}")

Rows after filtering to top 13 characters: 82871
Unique characters: 13


## 7. Aggregate per Question

Each visualization question wants a different grain. Pre-aggregating here keeps the visualization notebook fast and the chart specs simple.

| File    | Grain                                                              | Used by                               |
| ------- | ------------------------------------------------------------------ | ------------------------------------- |
| `df_q1` | (character, episode) — words & sentences per episode               | Q1: words per character               |
| `df_q2` | (character, season) — words & sentences per season                 | Q2: evolution across seasons          |
| `df_q3` | (character, season, episode-in-season) — words & sentences         | Q3: pair comparison within a season   |
| `df_q4` | (character, season, episode-in-season, minute) — words & sentences | Q4: pair comparison within an episode |
| `df_q5` | (character, episode) — sentences per episode                       | Q5: sentences per character           |

`fill_missing_combinations` ensures every character has a row for every grid cell (filled with 0 if they didn't speak), so charts have aligned bars/lines instead of irregular gaps. For Q4 this also matters visually: without zero-rows, minutes where neither selected character spoke would silently disappear from the ordinal x-axis, making the gap invisible to the reader.


In [7]:
def fill_missing_combinations(
    df, entity_cols, grid_cols, value_col, agg_func="sum", fill_value=0
):
    """
    Group `df` by entity + grid columns and ensure every entity has a row for
    every valid grid combination, filling missing combinations with `fill_value`.

    Used so per-character bars/lines stay aligned even when a character has zero
    words in some season/episode/minute.
    """
    if isinstance(entity_cols, str):
        entity_cols = [entity_cols]
    if isinstance(grid_cols, str):
        grid_cols = [grid_cols]
    all_group_cols = entity_cols + grid_cols
    grouped_df = df.groupby(all_group_cols)[value_col].agg(agg_func).reset_index()
    unique_entities = df[entity_cols].drop_duplicates()

    if len(grid_cols) > 2:
        # For 3+ grid cols (e.g. season → number_in_season → timestamp_in_min),
        # only keep grid combinations that actually exist (e.g. real episode minutes
        # within a real episode), avoiding a silly Cartesian product.
        parent_cols = grid_cols[:-1]
        target_col = grid_cols[-1]
        max_df = df.groupby(parent_cols)[target_col].max().reset_index()
        max_df[target_col] = max_df[target_col].apply(
            lambda x: list(range(1, int(x) + 1))
        )
        valid_grid_points = max_df.explode(target_col).reset_index(drop=True)
        valid_grid_points = valid_grid_points[grid_cols]
    else:
        valid_grid_points = df[grid_cols].drop_duplicates()

    master_grid = unique_entities.merge(valid_grid_points, how="cross")
    complete_df = master_grid.merge(grouped_df, on=all_group_cols, how="left")
    complete_df[value_col] = complete_df[value_col].fillna(fill_value)
    complete_df = complete_df.sort_values(by=all_group_cols).reset_index(drop=True)
    return complete_df


df_q1 = (
    df_final.groupby(["character_names", "episode_id"])[
        ["word_count", "sentence_count"]
    ]
    .sum()
    .reset_index()
)
# df_q2 / df_q3 / df_q4: aggregate both metrics so the dashboard's metric toggle
# can switch between word_count and sentence_count. Run fill_missing_combinations
# once per metric and merge on the entity + grid keys.
df_q2_words = fill_missing_combinations(
    df=df_final,
    entity_cols="character_names",
    grid_cols=["season"],
    value_col="word_count",
)
df_q2_sentences = fill_missing_combinations(
    df=df_final,
    entity_cols="character_names",
    grid_cols=["season"],
    value_col="sentence_count",
)
df_q2 = df_q2_words.merge(df_q2_sentences, on=["character_names", "season"])

df_q3_words = fill_missing_combinations(
    df=df_final,
    entity_cols="character_names",
    grid_cols=["season", "number_in_season"],
    value_col="word_count",
)
df_q3_sentences = fill_missing_combinations(
    df=df_final,
    entity_cols="character_names",
    grid_cols=["season", "number_in_season"],
    value_col="sentence_count",
)
df_q3 = df_q3_words.merge(
    df_q3_sentences, on=["character_names", "season", "number_in_season"]
)
# df_q4: aggregate to (character, season, episode, minute) and zero-fill so
# every minute of every episode has a row per character. This makes minutes
# where neither selected character spoke still appear as a tick on the x-axis
# (with no bar) instead of silently dropping out of the ordinal scale.
df_q4_words = fill_missing_combinations(
    df=df_final,
    entity_cols="character_names",
    grid_cols=["season", "number_in_season", "timestamp_in_min"],
    value_col="word_count",
)
df_q4_sentences = fill_missing_combinations(
    df=df_final,
    entity_cols="character_names",
    grid_cols=["season", "number_in_season", "timestamp_in_min"],
    value_col="sentence_count",
)
df_q4 = df_q4_words.merge(
    df_q4_sentences,
    on=["character_names", "season", "number_in_season", "timestamp_in_min"],
)
df_q5 = (
    df_final.groupby(["character_names", "episode_id"])["sentence_count"]
    .sum()
    .reset_index()
)

print(f"df_q1: {df_q1.shape}")
print(f"df_q2: {df_q2.shape}")
print(f"df_q3: {df_q3.shape}")
print(f"df_q4: {df_q4.shape}")
print(f"df_q5: {df_q5.shape}")


df_q1: (4723, 4)
df_q2: (325, 4)
df_q3: (7124, 5)
df_q4: (153543, 6)
df_q5: (4723, 3)


## 8. Verification & Save

Final sanity check: no nulls in any output, all dtypes are expected, then write the five CSVs to `data/clean_data/`.


In [8]:
for name, df in [
    ("df_q1", df_q1),
    ("df_q2", df_q2),
    ("df_q3", df_q3),
    ("df_q4", df_q4),
    ("df_q5", df_q5),
]:
    print(f"--- {name} ---")
    print(f"  shape: {df.shape}")
    print(f"  nulls: {df.isnull().sum().sum()}")
    print(f"  dtypes: {df.dtypes.to_dict()}")

df_q1.to_csv("data/clean_data/df_q1.csv", index=False)
df_q2.to_csv("data/clean_data/df_q2.csv", index=False)
df_q3.to_csv("data/clean_data/df_q3.csv", index=False)
df_q4.to_csv("data/clean_data/df_q4.csv", index=False)
df_q5.to_csv("data/clean_data/df_q5.csv", index=False)

print("\nSaved df_q1.csv … df_q5.csv to data/clean_data/")

--- df_q1 ---
  shape: (4723, 4)
  nulls: 0
  dtypes: {'character_names': dtype('O'), 'episode_id': dtype('int64'), 'word_count': dtype('float64'), 'sentence_count': dtype('int64')}
--- df_q2 ---
  shape: (325, 4)
  nulls: 0
  dtypes: {'character_names': dtype('O'), 'season': dtype('int64'), 'word_count': dtype('float64'), 'sentence_count': dtype('int64')}
--- df_q3 ---
  shape: (7124, 5)
  nulls: 0
  dtypes: {'character_names': dtype('O'), 'season': dtype('int64'), 'number_in_season': dtype('int64'), 'word_count': dtype('float64'), 'sentence_count': dtype('float64')}
--- df_q4 ---
  shape: (153543, 6)
  nulls: 0
  dtypes: {'character_names': dtype('O'), 'season': dtype('int64'), 'number_in_season': dtype('int64'), 'timestamp_in_min': dtype('O'), 'word_count': dtype('float64'), 'sentence_count': dtype('float64')}
--- df_q5 ---
  shape: (4723, 3)
  nulls: 0
  dtypes: {'character_names': dtype('O'), 'episode_id': dtype('int64'), 'sentence_count': dtype('int64')}

Saved df_q1.csv … df_q5.